# **Automated daily monitoring of API errors using Databricks & KQL**

*Purpose:* Query API errors from Azure Application Insights and analyze failure rates, error messages, and trends.

Install Azure Monitor and Identity libraries needed to connect to Application Insights.
- azure-monitor-query → sends KQL queries to Log Analytics
- azure-identity → authenticates using Service Principal

In [0]:
from datetime import datetime, timedelta
 
dbutils.widgets.text('run_date', '', 'Run Date (YYYY-MM-DD)')
run_date_input = dbutils.widgets.get('run_date')
 
if run_date_input == '':
    run_date = (datetime.utcnow() - timedelta(days=1)).strftime('%Y-%m-%d')
else:
    run_date = run_date_input
 
start_time = f'{run_date}T00:00:00Z'
end_time   = f'{run_date}T23:59:59Z'
print(f'Reporting for date: {run_date}')
print(f"start_time : {start_time}")
print(f"end_time   : {end_time}")


In [0]:
%pip install azure-monitor-query azure-identity

Connects to Azure using a **Service Principal** stored in Databricks Secrets.

| Secret Key | Description |
|---|---|
| tenant-id | Azure Directory ID |
| client-id | App Registration Client ID |
| client-secret | App Registration Secret |
| log-analytics-workspace-id | Log Analytics Workspace ID |

NOTE : Never hardcode credentials — always use dbutils.secrets.get()

In [0]:
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient

tenant_id=dbutils.secrets.get(scope="azure",key="tenant-id")
client_id=dbutils.secrets.get(scope="azure",key="client-id")
client_secret = dbutils.secrets.get(scope='azure', key='client-secret')
WORKSPACE_ID  = dbutils.secrets.get(scope='azure', key='log-analytics-workspace-id')

credential = ClientSecretCredential(tenant_id,client_id,client_secret)
client= LogsQueryClient(credential)

print('Connected to Application Insights')

Error Summary Query

Queries the AppRequests table for the failed requests

In [0]:
from azure.monitor.query import LogsQueryStatus
import pandas as pd

kql = f'''
AppRequests
| where TimeGenerated >= datetime('{start_time}')
  and   TimeGenerated <  datetime('{end_time}')
| where toint(ResultCode) >= 400
| summarize
    ErrorCount    = count(),
    AvgDurationMs = round(avg(DurationMs), 2),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by StatusCode = tostring(ResultCode),
     APIName    = tostring(Properties["API Name"]),
     Operation  = tostring(Properties["Operation Name"])
| order by ErrorCount desc
'''

resp = client.query_workspace(
    WORKSPACE_ID,
    kql,
    timespan=timedelta(hours=24)
)

if resp.status == LogsQueryStatus.SUCCESS:
    table = resp.tables[0]
    cols  = list(table.columns)          
    rows  = [dict(zip(cols, row)) for row in table.rows]
    print(f"Records found: {len(rows)}")
    if len(rows) == 0:
        print("No errors found for this date range")
    else:
        error_counts_df = spark.createDataFrame(pd.DataFrame(rows))
        error_counts_df.show(truncate=False)
else:
    print(f"Query failed: {resp.status}")

In [0]:
error_counts_df.display()

Error Detail Query

Fetches individual error records with full details including

In [0]:
kql_detail = f'''
AppRequests
| where TimeGenerated >= datetime('{start_time}')
  and   TimeGenerated <  datetime('{end_time}')
| where toint(ResultCode) >= 400
| extend APIType = iff(
    tostring(Properties["API Type"]) == "http",
    "rest",
    tostring(Properties["API Type"])
)
| join kind=leftouter (
    AppExceptions
    | project OperationId,
              ExceptionType = ExceptionType,
              ErrorMessage  = OuterMessage
) on OperationId
| project
    Timestamp     = TimeGenerated,
    APIName       = tostring(Properties["API Name"]),
    APIType,
    OperationName = tostring(Properties["Operation Name"]),
    StatusCode    = tostring(ResultCode),
    DurationMs    = round(DurationMs, 2),
    ExceptionType,
    ErrorMessage,
    ResponseBody  = tostring(Properties["Response-Body"])
| order by Timestamp desc
| take 1000
'''

resp2 = client.query_workspace(
    WORKSPACE_ID,
    kql_detail,
    timespan=timedelta(hours=24)
)

if resp2.status == LogsQueryStatus.SUCCESS:
    table2 = resp2.tables[0]
    cols2  = list(table2.columns)        
    rows2  = [dict(zip(cols2, row)) for row in table2.rows]

    print(f"Records found: {len(rows2)}")

    if len(rows2) == 0:
        print("No errors found for this date range")
    else:
        error_detail_df = spark.createDataFrame(pd.DataFrame(rows2))
        error_detail_df.createOrReplaceTempView("api_errors")
        display(error_detail_df)
else:
    print(f"Query failed: {resp2.status}")

Analysis & Failure Rate

Failure Rate is calculated by APIName and Operation

In [0]:
kql_all = f'''
AppRequests
| where TimeGenerated >= datetime('{start_time}')
  and   TimeGenerated <  datetime('{end_time}')
| extend APIType = iff(
    tostring(Properties["API Type"]) == "http",
    "rest",
    tostring(Properties["API Type"])
)
| project
    Timestamp     = TimeGenerated,
    APIName       = tostring(Properties["API Name"]),
    OperationName = tostring(Properties["Operation Name"]),
    APIType,
    StatusCode    = tostring(ResultCode),
    DurationMs    = DurationMs
'''

resp_all = client.query_workspace(
    WORKSPACE_ID,
    kql_all,
    timespan=timedelta(hours=24)
)

if resp_all.status == LogsQueryStatus.SUCCESS:
    table_all = resp_all.tables[0]
    cols_all  = list(table_all.columns)
    rows_all  = [dict(zip(cols_all, row)) for row in table_all.rows]

    all_requests_df = spark.createDataFrame(pd.DataFrame(rows_all))
    all_requests_df.createOrReplaceTempView("all_requests")
    print(f"Total requests found: {all_requests_df.count()}")
    display(all_requests_df)
else:
    print(f"Query failed: {resp_all.status}")

In [0]:
summary_df=spark.sql('''
    SELECT
        APIName,OperationName,
        COUNT(*) AS TotalRequests,
        SUM(CASE WHEN CAST(StatusCode AS INT) >= 400 THEN 1 ELSE 0 END) AS FailedRequests,
        SUM(CASE WHEN CAST(StatusCode AS INT) < 400  THEN 1 ELSE 0 END) AS SuccessRequests,
        ROUND(
            SUM(CASE WHEN CAST(StatusCode AS INT) >= 400 THEN 1 ELSE 0 END) 
            * 100.0 / COUNT(*), 
        2) AS FailureRatePercent
    FROM all_requests
    WHERE APIName IS NOT NULL
      AND APIName != ""
    GROUP BY APIName,OperationName
    ORDER BY FailureRatePercent DESC
''')


summary_df.display()

Unity Catalog Setup - API Error Monitoring

Note:
Prod support team has **READ ONLY** access

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS PROD_SUPPORT")
spark.sql("CREATE SCHEMA IF NOT EXISTS PROD_SUPPORT.APIM_MONITORING")

In [0]:
spark.sql("USE CATALOG PROD_SUPPORT")
spark.sql("USE SCHEMA APIM_MONITORING")

In [0]:
from pyspark.sql.functions import lit
 
error_detail_df_gold = error_detail_df.withColumn('report_date', lit(run_date))

 
error_detail_df_gold.write \
    .format('delta') \
    .mode('append') \
    .partitionBy('report_date', 'StatusCode') \
    .saveAsTable('azure_apim_daily_errors')
 
print(f'Saved {error_detail_df_gold.count()} records for {run_date}')


In [0]:
summary_df = summary_df.withColumn('report_date', lit(run_date)); run_datesummary_df_gold = summary_df


 
run_datesummary_df_gold.write \
    .format('delta') \
    .mode('append') \
    .partitionBy('report_date', 'APIName') \
    .saveAsTable('azure_apim_daily_errors_summary')
 
print(f'Saved {run_datesummary_df_gold.count()} records for {run_date}')

In [0]:
spark.sql("USE CATALOG PROD_SUPPORT")
spark.sql("USE SCHEMA APIM_MONITORING")

PROD_SUPPORT_GROUP = "prod-support@outlook.com"

spark.sql(f"GRANT USE CATALOG ON CATALOG PROD_SUPPORT TO `{PROD_SUPPORT_GROUP}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA APIM_MONITORING TO `{PROD_SUPPORT_GROUP}`")
spark.sql(f"GRANT SELECT ON TABLE azure_apim_daily_errors TO `{PROD_SUPPORT_GROUP}`")
spark.sql(f"GRANT SELECT ON TABLE azure_apim_daily_errors_summary TO `{PROD_SUPPORT_GROUP}`")

print(f"Permissions granted to {PROD_SUPPORT_GROUP}")